<a href="https://colab.research.google.com/github/vladigee/2026_Intro_Python/blob/main/live/part-I/1.2-data-structures-and-control-flow-environmental-exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> Notebook-friendly copy of `part-I/1.2-data-structures-and-control-flow-environmental-exercises.ipynb`, generated by `tools/make_live.py`. Edit the book notebook, not this file.

In [1]:
# --- environment setup (generated, not part of the lesson) ---
# Colab and Kaggle do not ship every package this notebook imports.
# This is a no-op in an environment that is already set up.
import importlib.util
import subprocess
import sys

for module, package in {"pooch": "pooch"}.items():
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

# Environmental Exercises

**ℹ️ How to use these exercises**

Attempt each from scratch in the empty code cell. Write plain functions with clear names, and avoid a mutable default argument.

## Exercise 1: A year of station data, continued

Exercises 1 and 2 are the main exercises of this subchapter.

In subchapter 1.1 you fetched this file, worked out its structure, and converted a single
record to °C. You did not yet have the tools to handle all of it at once — now you do.

The goal: turn a year of daily maximum temperatures into a monthly summary, and write that
summary out as a new file. The input file has the same three columns as in Exercise 1 of the
previous subchapter's environmental exercises: the station code, the day as `dd.mm.yy`, and the daily maximum temperature
in degrees Fahrenheit. Your monthly summary will have its own columns: `month`, `n_obs`
(the number of observations), and `mean_temp_celsius`.

In [2]:
# Pre-supplied: download the data file and cache it locally.
# You do not need to understand this cell yet — fetching data is covered in the
# reproducible-data-pipelines bonus subchapter.
import pooch

data_file = pooch.retrieve(
    url="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-I/station_iib_daily_max_temp_2022.csv",
    known_hash="sha256:034755fb289b4e157e5d029995481786a359eb25a80df62c09ee83d063013144",
    fname="station_iib_daily_max_temp_2022.csv",
    path=pooch.os_cache("mlees"),
)
print("data cached at:", data_file)

data cached at: C:\Users\redqu\AppData\Local\mlees\mlees\Cache\station_iib_daily_max_temp_2022.csv


**Step 1.** Write a function `fahrenheit_to_celsius(temp_fahrenheit)` that returns the
temperature in °C. Test it on the freezing point of water (32 °F should give 0.0 °C) and on
212 °F. A one-line docstring is optional here (see Exercise 11 of the practice exercises).

You wrote this conversion inline in 1.1. Steps 2 and 4 both need it again; with a function,
they call the same tested code instead of repeating the arithmetic.

In [2]:
# Your solution here
# Hint: °C = (°F - 32) * 5 / 9
# Hint (optional, see Exercise 11 of the practice exercises): a docstring goes on the first line of the function body,
# in triple quotes

def fahrenheit_to_celsius(temp_fahrenheit):
    temp_celsius = (temp_fahrenheit - 32) * 5 / 9
    return temp_celsius

fahrenh = 212
freezing_water_fahrenheit = 32

print(f"Temperatur: {fahrenheit_to_celsius(freezing_water_fahrenheit)} °C")
print(f"Temperatur: {fahrenheit_to_celsius(fahrenh)} °C")

Temperatur: 0.0 °C
Temperatur: 100.0 °C


**Step 2.** Open the file, read all lines, and loop over the records (skipping the header).
For each one, extract the month and the temperature in °C. Print the first three as a check.

Sixteen days in this file have an empty temperature field — a real gap in the station's record,
late October to mid-November. `float("")` raises `ValueError`, so your loop has to decide what to
do with those days before it can get to the end of the file. Skip them, and count how many you
skipped. After the loop, print how many records you converted and how many you skipped; the two
should add up to 365.

In [3]:
# Your solution here
# Hint: open it as in 1.1, with Path(data_file).open("r", encoding="utf-8") in a with block
# Hint: lines[1:] skips the header row
# Hint: .strip().split(",") gives you the three fields
# Hint: the month is the middle piece of the day field — split it on "."
# Hint: use a counter that you update with += inside the loop
# Hint: a day with no reading has fields[2] == "" — `continue` moves on to the next line

from pathlib import Path

data_file = r"C:\Users\redqu\AppData\Local\mlees\mlees\Cache\station_iib_daily_max_temp_2022.csv"
count_skipped = 0
count_converted = 0

with Path(data_file).open("r", encoding="utf-8") as fhandler:
    all_lines = fhandler.readlines()
    #print(all_lines[1:])
    for line in all_lines[1:]:
        cleaned_lines = line.strip().split(",")
        #print(cleaned_lines)
        date = cleaned_lines[1].split(".")
        #print(date)
        month = date[1]
        if cleaned_lines[2] == "":
            count_skipped += 1
            continue
        temp_fahrenheit = float(cleaned_lines[2])
        temp_celsius = fahrenheit_to_celsius(temp_fahrenheit)
        count_converted += 1
        if count_converted <= 3:
            print(month, temp_celsius)

print(f"Skipped: {count_skipped}")
print(f"Converted: {count_converted}")
print(f"Total: {count_skipped + count_converted}")

01 -9.61111111111111
01 -13.0
01 -4.222222222222223
Skipped: 16
Converted: 349
Total: 365


**Step 3.** Build a dictionary mapping each month to the number of observations in it.

You do not know in advance which months are present, so the dictionary has to grow as you go.
Keep skipping the days with no reading: a day the sensor did not record is not an observation, so
October and November should come out short of their full length.

In [4]:
# Your solution here
# Hint: start with an empty dict, counts = {}
# Hint: counts.get(month, 0) returns 0 for a month you have not seen yet
# Hint: counts[month] = counts.get(month, 0) + 1

counts = {}

for line in all_lines[1:]:
    cleaned_lines = line.strip().split(",")
    date = cleaned_lines[1].split(".")
    month = date[1]

    if cleaned_lines[2] == "":
        continue

    #counts.get(month, 0)
    counts[month] = counts.get(month, 0) + 1

print(counts)


{'01': 31, '02': 28, '03': 31, '04': 30, '05': 31, '06': 30, '07': 31, '08': 31, '09': 30, '10': 24, '11': 21, '12': 31}


**Step 4.** Build a second dictionary mapping each month to its **mean** daily maximum
temperature in °C. Print it with each mean rounded to two decimals.

In [ ]:
# Your solution here
# Hint: accumulate a running total per month in the same loop as the counts
# Hint: a second loop over the totals turns each total into a mean
# Hint: reuse your function from step 1 — do not repeat the conversion arithmetic

daily_max_temp_celsius = {}
monthly_means = {}

for line in all_lines[1:]:
    cleaned_lines = line.strip().split(",")
    date = cleaned_lines[1].split(".")
    month = date[1]
    if cleaned_lines[2] == "":
        continue
    temp_fahrenheit = float(cleaned_lines[2])
    temp_celsius = fahrenheit_to_celsius(temp_fahrenheit)
    daily_max_temp_celsius[month] = daily_max_temp_celsius.get(month, 0) + temp_celsius
for month in daily_max_temp_celsius:
    mean_temp = daily_max_temp_celsius[month]/counts[month]
    monthly_means[month] = mean_temp
for month, mean in monthly_means.items():
    print(f"{month}: {mean:.2f}")


01: -4.36
02: -0.22
03: 8.11
04: 11.55
05: 22.22
06: 27.45
07: 27.97
08: 26.68
09: 23.76
10: 17.76
11: 4.40
12: -1.39


**Step 5.** Find the warmest month and print it with its mean temperature.

Do this with a loop and a comparison, keeping track of the best value seen so far, as you did for
the warmest day in Exercise 6 of the practice exercises. A dictionary is looked up by key, not by position, so there is no
`[0]` to start from: start from `None` instead.

In [41]:
# Your solution here
# Hint: initialise warmest_month = None and warmest_mean = None before the loop
# Hint: if warmest_mean is None, take this month; elif its mean is larger, take it instead

warmest_month = None
warmest_mean = None

for month, mean in monthly_means.items():
    if warmest_month is None:
        warmest_month = month
        warmest_mean = mean
    elif mean > warmest_mean:
        warmest_mean = mean
        warmest_month = month

print(warmest_month, warmest_mean)

07 27.974910394265226


**Step 6.** Write your monthly summary to `_files/monthly_means.csv`, with the header
`month,n_obs,mean_temp_celsius` and one line per month, creating the `_files` folder first as you
did in 1.1. Then read the file back and print it, to confirm it says what you meant it to say.

In [49]:
# Your solution here
# Hint: out_path = Path("_files") / "monthly_means.csv" builds the path, and
# out_path.parent.mkdir(exist_ok=True) creates the _files folder if it is missing
# Hint: open with mode "w" inside a with block, and write the header first
# Hint: build each line with an f-string, and remember the "\n"
# Hint: f"{value:.2f}" keeps the file tidy

out_path = Path("_files") / "monthly_means.csv"
out_path.parent.mkdir(exist_ok=True)

with out_path.open("w", encoding="utf-8") as fhandler:
    header = fhandler.write(f"month,n_obs,mean_temp_celsius\n")
    for month in monthly_means:
        content = fhandler.write(f"{month},{counts[month]},{monthly_means[month]:.2f}\n")

with out_path.open("r", encoding="utf-8") as fhandler:
    read = fhandler.read()
    print(read)


month,n_obs,mean_temp_celsius
01,31,-4.36
02,28,-0.22
03,31,8.11
04,30,11.55
05,31,22.22
06,30,27.45
07,31,27.97
08,31,26.68
09,30,23.76
10,24,17.76
11,21,4.40
12,31,-1.39



**Step 7 (optional).** This step uses `try`/`except` from the going-deeper box in the lecture. Real files contain junk, and you have already met one kind of it — the empty
temperature field. Suppose some lines are truncated as well, or hold a temperature that is not a
number at all. Modify your reading loop so that it skips any line that does not have three fields
or whose temperature will not convert to a number, counting how many lines it skipped.

Test it on this list before applying it to the real file:

```python
test_lines = [
    "IIB,01.01.22,30.2",
    "IIB,02.01.22",
    "IIB,03.01.22,notanumber",
    "IIB,04.01.22,33.4",
]
```

You should keep 2 and skip 2.

In [ ]:
# Your solution here
# Hint: len(fields) != 3 catches the truncated line — use continue
# Hint: wrap the float() call in try/except ValueError to catch the bad number
# Hint: continue inside a for loop moves straight to the next line

**ℹ️ What you just did**

You wrote a reusable function, looped over a whole year of records, built dictionaries whose
keys you did not know in advance, aggregated by group, and wrote a derived dataset back to disk
— with a loop that survives malformed input. This is the shape of nearly every data-processing
script you will write. In subchapter 1.5, `pandas` will do the grouping in one line; the
reasoning will be exactly the same.

## Exercise 2: The solar system

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/nasa-solar-system-montage.jpg" alt="The eight planets and the Moon photographed by spacecraft, arranged diagonally against black" width="100%">

<em>The eight planets and the Moon, as photographed by Mariner 10, Magellan, Galileo, Mars Global
Surveyor, Cassini and the Voyager spacecraft. The inner bodies are roughly to scale with each other, as are the outer ones, but the
two groups are not to scale with one another. Image
<a href="https://www.jpl.nasa.gov/images/pia03153-solar-system-montage-high-resolution-2001-version/">NASA/JPL (PIA03153)</a>, public domain.</em>


This is the long exercise for this subchapter, and the only one in it whose data is not
environmental. Eight planets and their masses are few enough to type by hand, and familiar enough
that a wrong answer is easy to spot, which makes them a good first test of lists, dictionaries and
functions together.

Work through sections A to C in order: section C writes functions against the dictionary you
build in section B. You will pick the exercise up again in 1.7, once classes are available, and
turn the dictionary into a `Planet` type.

The masses come from NASA's
[planetary fact sheet](https://nssdc.gsfc.nasa.gov/planetary/factsheet/). Use units of
10^24 kg throughout, so that Earth is 5.97 and Jupiter is 1898 — the variable names below carry
that unit, which is what makes a bare number like `1898` readable three cells later.

### A: lists and loops

**Q1) Create a list with the names of every planet in the solar system, in order.**

Name it `planets` and use lowercase names, so it starts `["mercury", "venus", ...]`.

In [88]:
# Q1: create your list here

planets = ["mercury", "venus", "earth", "mars", "jupiter", "saturn", "uranus", "neptune"]

**Q2) Have Python tell you how many planets there are by examining your list.**

You should get 8. Do not type the number — read it off the list.

In [89]:
# Q2: count the planets
# Hint: len() reads the length off the list itself
print(len(planets))

8


**Q3) Use slicing to display the first four planets — the rocky ones.**

In [90]:
# Q3: the rocky planets

first_four = planets[0:4]
print(first_four)

['mercury', 'venus', 'earth', 'mars']


**Q4) Iterate through your planets and print the planet name only if it ends in "s".**

You should see `venus`, `mars` and `uranus`, and nothing else.

Hint: the last letter of a name is `planet[-1]`. Use `continue` for the names that do not match,
so the loop body says explicitly what it skips.

In [91]:
# Q4: print the planets whose name ends in s
#print(planets)
for planet in planets:
    if planet[-1] == "s":
        print(planet)
    elif planet[-1] != "s":
        continue

venus
mars
uranus


### B: dictionary

**Q5) Create a dictionary that maps each planet name to its mass.**

Call it `planet_masses_1e24kg`, and read the values off the
[NASA fact sheet](https://nssdc.gsfc.nasa.gov/planetary/factsheet/) in units of 10^24 kg. Every
value has to be in that one unit: entered as 1.898 (in 10^27 kg), Jupiter would come out 1000
times too light in every comparison and ratio below.

In [92]:
# Q5: build the dictionary here

planet_masses_1e24kg = {}
masses = [0.330,4.87,5.97,0.642,1898,568,86.8,102]
for p, m in zip(planets, masses):
    planet_masses_1e24kg[p] = m

print(planet_masses_1e24kg.items())

dict_items([('mercury', 0.33), ('venus', 4.87), ('earth', 5.97), ('mars', 0.642), ('jupiter', 1898), ('saturn', 568), ('uranus', 86.8), ('neptune', 102)])


**Q6) Use your dictionary to look up Earth's mass.**

In [93]:
# Q6: look up Earth
print(planet_masses_1e24kg.get("earth"))

5.97


**Q7) Loop through the dictionary and build a list of every planet heavier than
100 x 10^24 kg.**

You should end up with `["jupiter", "saturn", "neptune"]` — uranus, at 86.8, does not make it.

Hint: `.items()` gives you the name and the mass together on each pass, as in 1.2's lecture.

In [94]:
# Q7: the planets heavier than 100e24 kg

heavy = []

for p,m in planet_masses_1e24kg.items():
    if m > 100:
        heavy.append(p)
    #else:
        #continue

print(heavy)

['jupiter', 'saturn', 'neptune']


**Q8) Add Pluto to your dictionary.**

Its mass is 0.0130 x 10^24 kg. Print the names afterwards, looping with `.items()`, to confirm it
landed.

Pluto has not been a planet since 2006. Adding it still takes one assignment, and none of the code
that reads the other eight entries has to change.

In [96]:
# Q8: add Pluto, then print the names

planet_masses_1e24kg["pluto"] = 0.0130
print(planet_masses_1e24kg.keys())

dict_keys(['mercury', 'venus', 'earth', 'mars', 'jupiter', 'saturn', 'uranus', 'neptune', 'pluto'])


### C: functions

**Q9) Write a function that converts a planetary mass to Earth masses.**

Jupiter is about 1898 x 10^24 kg, which is about 318 Earth masses:

$$M_{\text{Jupiter}} \approx 1898 \times 10^{24}\,\text{kg} \approx 318\,M_{\text{Earth}}$$

Call it `to_earth_masses(mass_1e24kg)` and check it against Jupiter — you should get
317.92294807370183.

In [99]:
# Q9: write to_earth_masses here, then test it on Jupiter

def to_earth_masses(mass_1e24kg):
    earth_equivalent = mass_1e24kg/planet_masses_1e24kg["earth"]
    return earth_equivalent

to_earth_masses(planet_masses_1e24kg["jupiter"])

317.92294807370183

**Q10) Now write a single function that converts to either Earth masses or Jupiter
masses, depending on a keyword argument.**

Call it `to_planet_masses(mass_1e24kg, reference="jupiter")`. Converting Jupiter's own mass should
give 317.923 with `reference="earth"` and exactly 1.0 with `reference="jupiter"`.

With the default argument, one function covers both conversions. Jupiter in Jupiter masses has
to come out as exactly 1.0; any other value means the function divides by the wrong mass.

In [103]:
# Q10: write to_planet_masses here, then test both references on Jupiter

def to_planet_masses(mass_1e24kg, reference="jupiter"):
    if reference == "jupiter":
        planet_mass = mass_1e24kg/planet_masses_1e24kg["jupiter"]
    elif reference == "earth":
        planet_mass = mass_1e24kg/planet_masses_1e24kg["earth"]
    return planet_mass

print(to_planet_masses(planet_masses_1e24kg["jupiter"],"earth"))
print(to_planet_masses(planet_masses_1e24kg["jupiter"]))

317.92294807370183
1.0


**Q11) Write a function that returns two values.**

`mass_in_two_units(mass_1e24kg, reference)` should return the mass in Earth masses *and* in masses
of whatever reference planet you pass. Test it on Jupiter with `reference="mars"`.

Hint: `return a, b` returns a tuple, and `first, second = f(...)` unpacks it — both from 1.2's
lecture.

In [107]:
# Q11: write mass_in_two_units here, then test it on Jupiter against mars
def mass_in_two_units(mass_1e24kg, reference):
    in_earth_mass = mass_1e24kg/planet_masses_1e24kg["earth"]
    in_reference_mass = mass_1e24kg/planet_masses_1e24kg[reference]
    return in_earth_mass, in_reference_mass

print(mass_in_two_units(planet_masses_1e24kg["jupiter"], reference="mars"))

(317.92294807370183, 2956.386292834891)


**Bonus.** Convert Neptune's mass to Jupiter masses with your Q10 function. Then pass that result
to your Q11 function with `reference="jupiter"`, and compare the second value it returns with the
Q10 result. Are they the same? If not, work out which function is being given a number in the
wrong unit, and what it should have been given instead — this is the mistake the `_1e24kg` suffix
exists to prevent.

In [ ]:
# Bonus: Neptune, out and back
neptune_to_jupiter_mass = to_planet_masses(planet_masses_1e24kg["neptune"])
print(neptune_to_jupiter_mass)
print(mass_in_two_units(neptune_to_jupiter_mass, "jupiter"))

# Logic error: Mixing up units
# First, I converted Neptune's mass from (10^{24}) kg into Jupiter masses using `to_planet_masses()`.
# Then, I passed this converted value to `mass_in_two_units()`, which expects its input in units of 
# (10^{24}) kg, not Jupiter masses.
# The function therefore divides a value that is already expressed in Jupiter masses by Jupiter's 
# mass again. This produces an incorrect result.
# Solution: To obtain the correct result, I should pass Neptune's original mass 
# (`planet_masses_1e24kg["neptune"]`) to `mass_in_two_units()`.
# Lesson learned: This is a logic error rather than a syntax error. 
# Python executes the code without problems, but the result is scientifically incorrect 
# because the units are inconsistent. The `_1e24kg` suffix helps indicate which units a function expects.
# To avoid the logic error, I could use a suffix in the variable name
# to indicate the unit, e.g. _1e24kg or _jupiter_masses.
# This would make it clear that neptune_jupiter_masses is already
# converted and should not be passed to a function expecting 1e24 kg.
# The suffix does not prevent errors automatically, but it helps
# identify unit mismatches before running the code.

0.05374077976817703
(0.009001805656311061, 2.831442558913437e-05)


**ℹ️ What you just did**

You built a list and read its length rather than counting it, sliced a subset out of it, and
looped over it with a condition. You built a dictionary keyed by name, looked a value up, filtered
it into a new list, and extended it. Then you wrote three functions over that dictionary: one
fixed, one with a default argument that collapses two functions into one, and one returning a pair.

That is the whole of this subchapter applied to one dataset. In 1.7 you will come back and replace
the dictionary with a class, so that a planet's name and mass travel together as one object
instead of as a key and a value that happen to line up.